# Preprocessing check

A check on the output of `data_prep.py`, confirming that the cleaning rules and feature matrices match what section 8 of the EDA expects.


In [1]:
# Environment

from pathlib import Path

import numpy as np
import pandas as pd

import data_prep as dp

DATA_DIR = Path(".")
#OUT_DIR = Path("prep_output")
#OUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["CompanyNumber_norm", "period_t", "period_t_plus_1"]
DATE_COLS = ["period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1"]

In [2]:
# Load and join metadata
pairs = pd.read_csv(DATA_DIR / "04_Five_CSV/03_financial_change_labels.csv",
                    dtype={"CompanyNumber_norm": str}, low_memory=False)
for c in DATE_COLS:
    pairs[c] = pd.to_datetime(pairs[c], errors="coerce")

meta = pd.read_csv(DATA_DIR / "01_CompaniesSelected/UKcompanies_active_account_category_sample_100k.csv",
                   dtype={"CompanyNumber": str}, low_memory=False)
meta["CompanyNumber_norm"] = meta["CompanyNumber"].map(dp.normalise_company_number)
meta["IncorporationDate"] = pd.to_datetime(meta["IncorporationDate"], errors="coerce")

META_KEEP = ["CompanyNumber_norm", "IncorporationDate", "CompanyCategory", "multi_sic_company"]
raw = pairs.merge(meta[META_KEEP], on="CompanyNumber_norm",
                  how="left", validate="many_to_one")

print(f"rows: {len(raw):,}")
print(f"companies: {raw['CompanyNumber_norm'].nunique():,}")
print(f"unmatched metadata: {raw['IncorporationDate'].isna().sum():,}")

rows: 81,603
companies: 76,611
unmatched metadata: 0


## 1. Cleaning log

Compared against the EDA figures

In [3]:
df, log = dp.clean_global(raw)

print("cleaning log:")
print(log.to_string())

# totals by rule, restricted to the _t columns for comparability
t_only = log[log["column"].str.endswith("_t")]
print("\n_t totals on _t columns:")
print(t_only.groupby("rule")["n"].sum().to_string())

cleaning log:
                     rule                    column    n
0         negative_to_nan          current_assets_t  155
1         negative_to_nan   current_assets_t_plus_1  186
2         negative_to_nan            fixed_assets_t   47
3         negative_to_nan     fixed_assets_t_plus_1   72
4         negative_to_nan         creditors_total_t  812
5         negative_to_nan  creditors_total_t_plus_1  877
6         negative_to_nan                    cash_t   92
7         negative_to_nan             cash_t_plus_1  116
8         negative_to_nan                 debtors_t  153
9         negative_to_nan          debtors_t_plus_1  161
10        negative_to_nan               employees_t  605
11        negative_to_nan        employees_t_plus_1  195
12  employees_implausible               employees_t   33
13  employees_implausible        employees_t_plus_1   30

_t totals on _t columns:
rule
employees_implausible      33
negative_to_nan          1864


In [4]:
# Merged category distribution
'''EDA(train):
MICRO              36048
TEF                24839
ABRIDGED            3178
FULL_DISCLOSURE     1074
GROUP                154
'''

print(df["acct_cat_model"].value_counts().to_string())
print(f"\nraw categories: {df['acct_cat_raw'].nunique()} - merged: {df['acct_cat_model'].nunique()}")

# sanity of the derived columns
print(f"\nis_private_limited   : {df['is_private_limited'].mean():.4f}")
print(f"company_age_at_t missing: {df['company_age_at_t'].isna().sum():,}")
print(f"company_age_at_t negative: {(df['company_age_at_t'] < 0).sum():,}")
print(f"period_month range: {df['period_month'].min()}~{df['period_month'].max()}")

acct_cat_model
MICRO              45180
TEF                30977
ABRIDGED            3911
FULL_DISCLOSURE     1351
GROUP                184

raw categories: 8 - merged: 5

is_private_limited   : 0.9691
company_age_at_t missing: 0
company_age_at_t negative: 0
period_month range: 1~12


## 2. Feature matrices
The GBM branch should retain NaN, since trees handle missingness natively and imputing would
conflate a genuine zero with an undisclosed value. 

The Ridge branch should contain no NaN after imputation, and have many more columns: split bimodal fields, one-hot encoding and interactions.

In [5]:
# Load the split assignment
assignment = pd.read_csv("./eda_output/split_assignment.csv",
                         dtype={"CompanyNumber_norm": str},
                         parse_dates=["period_t", "period_t_plus_1"])
df = df.merge(assignment, on=KEY, how="left", validate="one_to_one")
train = df[df["split_company"] == "train"].copy()
print(f"train: {len(train):,} rows")

results = {}
for variant in ("gbm", "ridge"):
    X, cat_cols, stats = dp.build_matrix(train, variant)
    results[variant] = (X, cat_cols, stats)
    n_const = int((X.select_dtypes("number").nunique() <= 1).sum())
    print(f"\n{variant}: columns={X.shape[1]}  "
          f"categorical={len(cat_cols)}  "
          f"columns with NaN={int(X.isna().any().sum())}  "
          f"constant={n_const}")
    print(f"fitted stats: {sorted(stats) or 'none'}")

train: 65,293 rows

gbm: columns=30  categorical=3  columns with NaN=11  constant=0
fitted stats: none

ridge: columns=134  categorical=0  columns with NaN=0  constant=0
fitted stats: ['cat_levels', 'impute', 'impute_simple', 'winsor']


In [6]:
# Structural checks
X_gbm, cat_gbm, _ = results["gbm"]
X_ridge, _, stats_ridge = results["ridge"]

# GBM: missingness should be retained and match the source fields
print("GBM missing rate:")
for m in dp.METRICS[:5]:
    print(f"  {m:38s} X={X_gbm[f'{m}_sl'].isna().mean():.4f}  "
          f"source={train[f'{m}_t'].isna().mean():.4f}")

# Ridge: no NaN
n_nan = int(X_ridge.isna().sum().sum())
print(f"\nRidge remaining NaN: {n_nan}")
if n_nan:
    print(X_ridge.columns[X_ridge.isna().any()].tolist())

# Imputation mode: group median versus global fallback
modes = pd.Series({c: k for c, (k, _, _) in stats_ridge["impute"].items()})
print(f"\nimputation mode:\n{modes.value_counts().to_string()}")
print("columns falling back to global:")
print("  " + ", ".join(modes[modes == "global"].index) if (modes == "global").any() else "  none")

# interaction terms
inter = [c for c in X_ridge.columns if "__x__" in c]
print(f"\ninteractions: {len(inter)}  "
      f"all-zero: {int((X_ridge[inter] == 0).all().sum())}")

GBM missing rate:
  current_assets                         X=0.1222  source=0.1222
  fixed_assets                           X=0.4761  source=0.4761
  creditors_total                        X=0.0762  source=0.0762
  equity                                 X=0.0294  source=0.0294
  net_assets_liabilities                 X=0.1629  source=0.1629

Ridge remaining NaN: 0

imputation mode:
group    17
columns falling back to global:
  none

interactions: 85  all-zero: 0


## 3. Within-group percentile
The optional feature used in the ablation. It is a statistic across rows. Whether it adds anything is what the ablation tests.

In [ ]:
X_pp, _, stats_pp = dp.build_matrix(train, "gbm", peer_pct=True)
pp_cols = [c for c in X_pp.columns if c.endswith("_peer_pct")]

print(f"added columns: {len(pp_cols)}\n")

cov = pd.DataFrame({
    "peer_pct_missing": X_pp[pp_cols].isna().mean().values,
    "source_missing": [train[f"{c.replace('_peer_pct', '')}_t"].isna().mean()
                       for c in pp_cols],
}, index=[c.replace("_peer_pct", "") for c in pp_cols])
cov["gap"] = (cov["peer_pct_missing"] - cov["source_missing"]).round(4)
print("gap should be zero: percentile missingness should stem only from the source\n")
print(cov.round(4).to_string())


added columns: 11

gap should be near zero: percentile missingness should stem only from the source

                                       peer_pct_missing  source_missing  gap
current_assets                                   0.1222          0.1222  0.0
fixed_assets                                     0.4761          0.4761  0.0
creditors_total                                  0.0762          0.0762  0.0
equity                                           0.0294          0.0294  0.0
net_assets_liabilities                           0.1629          0.1629  0.0
net_current_assets_liabilities                   0.0724          0.0724  0.0
cash                                             0.5787          0.5787  0.0
debtors                                          0.6443          0.6443  0.0
employees                                        0.0544          0.0544  0.0
profit_loss                                      0.9496          0.9496  0.0
total_assets_less_current_liabilities            0.1